In [1]:
"""
Figure 6: Surface Energy Budget and Cold Air Outbreak Index
============================================================
Two-panel figure showing:
(a) Total surface energy flux (early and late winter)
(b) Cold Air Outbreak (CAO) index (early and late winter)

Surface energy budget: F0 = FSW_net + FLW_net + FSH + FLH
CAO Index = θ_skin - θ_atmosphere

Analysis split into:
- Early Winter (ONDJ): October, November, December, January
- Late Winter (FMA): February, March, April

Trend analysis split into two periods:
- Period 1: 1979-2015
- Period 2: 2015-2025

Statistical testing includes:
- Mann-Kendall trend test
- Bootstrap confidence intervals (95%)

Dataset: ERA5 Reanalysis (1979-2025)
Region: Greenland Sea
Version: 2.1.0
Last Modified: 2026-06-22
Author: Chris Barrell

"""

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy import stats as scipy_stats
import shapely.geometry as sgeom
import regionmask
import metpy.calc as mpcalc
from metpy.units import units
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Setup logging
import sys
sys.path.append('..')
from utils.logger import (setup_logger, log_data_loading, log_processing_step,
                          log_output_file, log_completion, log_error)

logger = setup_logger('figure_06', config_path='../config.yaml')
start_time = datetime.now()

try:
    # ====================================================================
    # CONFIGURATION
    # ====================================================================

    logger.info("Loading configuration...")

    # Data paths
    DATA_DIR = Path('../../era5/')
    DATA_PATTERN = 'era5_*_Arctic.nc'
    OUTPUT_DIR = Path('./outputs/figures/')
    PROCESSED_DIR = Path('./outputs/processed_data/figure_06')
    OUTPUT_METHODS_DIR = Path('./outputs/methods')

    # Create output directories
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_METHODS_DIR.mkdir(parents=True, exist_ok=True)

    # Greenland Sea polygon coordinates
    GREENLAND_SEA_COORDS = [
        (-22, 71), (-8.5, 71), (12, 79), (-21, 79), (-28, 73), (-22, 71)
    ]

    # Winter period definitions
    EARLY_WINTER_MONTHS = [10, 11, 12, 1]  # ONDJ
    LATE_WINTER_MONTHS = [2, 3, 4]         # FMA

    # Configuration options
    PRESSURE_LEVEL = 850  # hPa
    SIC_THRESHOLD = 0.15  # 15%

    # Split-trend analysis periods
    SPLIT_YEAR = 2015
    PERIOD1_YEARS = (1979, 2015)
    PERIOD2_YEARS = (2015, 2025)

    # Color scheme
    COLOR_EARLY = '#3D3D3D'  # charcoal
    COLOR_LATE = '#B8960C'   # gold

    # Figure size — matches GLORYS T/S timeseries figure
    FIGURE_SIZE_TIMESERIES = (8, 9)

    # Display options
    SHOW_STATS = True
    SAVE_DATA = True
    N_BOOTSTRAP = 1000
    DPI = 600
    REPROCESS = False  # Set True to regenerate CSVs from scratch

    logger.info(f"Split year: {SPLIT_YEAR}")
    logger.info(f"SIC threshold: {SIC_THRESHOLD}")
    logger.info(f"Pressure level: {PRESSURE_LEVEL} hPa")

    # ====================================================================
    # UTILITY FUNCTIONS
    # ====================================================================

    def get_seconds_in_month(time_coord):
        """Calculate number of seconds in each month."""
        times = pd.to_datetime(time_coord.values)
        seconds = []
        for t in times:
            days_in_month = pd.Period(t, freq='M').days_in_month
            seconds.append(days_in_month * 24 * 3600)
        return np.array(seconds)


    def convert_accumulated_to_rate(data, time_coord):
        """Convert daily accumulated flux (J/m²) to flux rate (W/m²)."""
        times = pd.to_datetime(time_coord.values)

        days_per_month = []
        seconds_per_month = []
        for t in times:
            days = pd.Period(t, freq='M').days_in_month
            days_per_month.append(days)
            seconds_per_month.append(days * 24 * 3600)

        days_per_month = np.array(days_per_month)
        seconds_per_month = np.array(seconds_per_month)

        time_dim = data.dims[0]

        days_da = xr.DataArray(days_per_month, dims=[time_dim], coords={time_dim: time_coord})
        seconds_da = xr.DataArray(seconds_per_month, dims=[time_dim], coords={time_dim: time_coord})

        data_rate = (data * days_da) / seconds_da
        return data_rate


    def mann_kendall_test(years, values):
        """Perform Mann-Kendall trend test."""
        valid = ~np.isnan(values)
        if valid.sum() < 3:
            return None

        years_clean = years[valid]
        values_clean = values[valid]

        tau, p_value = scipy_stats.kendalltau(years_clean, values_clean)
        return {'tau': tau, 'p_value': p_value}


    def bootstrap_confidence_interval(years, values, n_boot=1000, ci=95):
        """Calculate bootstrap confidence intervals."""
        valid = ~np.isnan(values)
        if valid.sum() < 3:
            return None

        years_clean = years[valid]
        values_clean = values[valid]
        n = len(values_clean)

        slopes = np.zeros(n_boot)
        predictions = np.zeros((n_boot, n))

        for i in range(n_boot):
            indices = np.random.choice(n, size=n, replace=True)
            y_boot = values_clean[indices]
            x_boot = years_clean[indices]

            slope, intercept, _, _, _ = scipy_stats.linregress(x_boot, y_boot)
            slopes[i] = slope
            predictions[i, :] = slope * years_clean + intercept

        alpha = (100 - ci) / 2
        ci_lower_slope = np.percentile(slopes, alpha)
        ci_upper_slope = np.percentile(slopes, 100 - alpha)

        ci_lower_band = np.percentile(predictions, alpha, axis=0)
        ci_upper_band = np.percentile(predictions, 100 - alpha, axis=0)

        return {
            'ci_lower': ci_lower_slope,
            'ci_upper': ci_upper_slope,
            'ci_lower_band': ci_lower_band,
            'ci_upper_band': ci_upper_band,
            'slopes': slopes
        }


    def get_trend_stats(years, values, period_label=''):
        """Calculate comprehensive trend statistics."""
        valid = ~np.isnan(values)
        if valid.sum() < 3:
            return None

        years_clean = years[valid]
        values_clean = values[valid]

        slope, intercept, r_value, p_value, std_err = scipy_stats.linregress(
            years_clean, values_clean
        )

        trend_line = slope * years_clean + intercept
        mk_result = mann_kendall_test(years_clean, values_clean)
        bootstrap_ci = bootstrap_confidence_interval(years_clean, values_clean, n_boot=N_BOOTSTRAP)

        if period_label:
            logger.info(f"    {period_label}:")
            logger.info(f"      Linear trend: {slope:.4f} /yr (p={p_value:.4f})")
            if mk_result:
                logger.info(f"      Mann-Kendall: tau={mk_result['tau']:.3f}, p={mk_result['p_value']:.4f}")
            if bootstrap_ci:
                logger.info(f"      95% CI: [{bootstrap_ci['ci_lower']:.4f}, {bootstrap_ci['ci_upper']:.4f}] /yr")

        return {
            'slope': slope,
            'intercept': intercept,
            'r_value': r_value,
            'p_value': p_value,
            'std_err': std_err,
            'trend_line': trend_line,
            'years': years_clean,
            'mk_result': mk_result,
            'bootstrap_ci': bootstrap_ci
        }


    def create_greenland_sea_mask(ds, lon_name='longitude', lat_name='latitude'):
        """Create Greenland Sea mask."""
        greenland_sea_polygon = sgeom.Polygon(GREENLAND_SEA_COORDS)
        region = regionmask.Regions(
            [greenland_sea_polygon],
            names=["Greenland Sea"],
            abbrevs=["GS"]
        )

        mask = region.mask(ds[lon_name], ds[lat_name])
        mask_bool = (mask == 0)
        mask_bool.name = "greenland_sea_mask"

        return mask_bool, region


    logger.info("Functions defined")

    # ====================================================================
    # DATA LOADING
    # ====================================================================

    def load_era5_data():
        """Load preprocessed ERA5 data."""
        logger.info("Loading preprocessed ERA5 data...")

        file_list = sorted(DATA_DIR.glob(DATA_PATTERN))
        logger.info(f"  Found {len(file_list)} files")

        if len(file_list) == 0:
            raise FileNotFoundError(f"No files found matching {DATA_DIR / DATA_PATTERN}")

        ds_era = xr.open_mfdataset(
            file_list,
            chunks='auto',
            combine='by_coords',
            parallel=False
        )

        logger.info(f"  Time range: {ds_era.time.values[0]} to {ds_era.time.values[-1]}")

        required_vars = ['skt', 't', 'siconc', 'lsm', 'ssr', 'ssrd', 'str', 'strd', 'sshf', 'slhf']
        missing_vars = [var for var in required_vars if var not in ds_era.data_vars]

        if missing_vars:
            raise ValueError(f"Missing variables: {missing_vars}")

        logger.info("  All required variables present!")
        return ds_era


    # ====================================================================
    # SURFACE ENERGY BUDGET CALCULATION
    # ====================================================================

    def calculate_energy_budget(ds_era):
        """Calculate total surface energy budget."""
        logger.info("Calculating surface energy budget...")

        mask_gs, region = create_greenland_sea_mask(ds_era)

        ds_gs = ds_era.where(mask_gs, drop=True)
        ds_gs_sea = ds_gs.where(ds_gs.lsm == 0)
        ds_gs_sea = ds_gs_sea.where(ds_gs_sea.siconc < SIC_THRESHOLD)

        logger.info("  Converting accumulated fluxes to rates...")

        sw_net = convert_accumulated_to_rate(ds_gs_sea.ssr, ds_gs_sea.time)
        lw_net = convert_accumulated_to_rate(ds_gs_sea.str, ds_gs_sea.time)
        sensible = convert_accumulated_to_rate(ds_gs_sea.sshf, ds_gs_sea.time)
        latent = convert_accumulated_to_rate(ds_gs_sea.slhf, ds_gs_sea.time)

        total = sw_net + lw_net + sensible + latent

        total_mean = total.mean(dim=['longitude', 'latitude'])
        total_mean.attrs['units'] = 'W/m²'

        logger.info(f"  Mean total flux: {float(total_mean.mean()):.2f} W/m²")

        return total_mean


    # ====================================================================
    # CAO INDEX CALCULATION
    # ====================================================================

    def calculate_cao_index(ds_era, pressure_level):
        """Calculate Cold Air Outbreak index."""
        logger.info(f"Calculating CAO index for {pressure_level} hPa...")

        mask_gs, region = create_greenland_sea_mask(ds_era)
        ds_gs = ds_era.where(mask_gs, drop=True)
        ds_gs_sea = ds_gs.where(ds_gs.lsm == 0)
        ds_gs_sea = ds_gs_sea.where(ds_gs_sea.siconc < SIC_THRESHOLD)

        T_plev = ds_gs_sea.t.sel(pressure_level=pressure_level)

        theta_plev = xr.DataArray(
            mpcalc.potential_temperature(
                pressure_level * units.mbar,
                T_plev.values * units.kelvin
            ).magnitude,
            coords=T_plev.coords,
            dims=T_plev.dims
        )

        cao_index = ds_gs_sea.skt - theta_plev
        cao_mean = cao_index.mean(dim=['longitude', 'latitude'])
        cao_mean.attrs['units'] = 'K'

        logger.info(f"  Mean CAO index: {float(cao_mean.mean()):.2f} K")

        return cao_mean


    # ====================================================================
    # SEASONAL AGGREGATION
    # ====================================================================

    def calculate_seasonal_means(data, season_months, season_name):
        """Calculate seasonal mean."""
        logger.info(f"  Calculating {season_name} winter seasonal means...")

        def get_season_year(time, is_early):
            year = time.dt.year
            month = time.dt.month

            if is_early:
                return xr.where((month >= 10), year + 1, year)
            else:
                return year

        season_mask = data.time.dt.month.isin(season_months)
        is_early = (season_name.lower() == 'early')
        n_months_expected = len(season_months)

        season_data = data.sel(time=season_mask)
        season_data = season_data.assign_coords(
            season_year=('time', get_season_year(season_data.time, is_early).data)
        )

        months_per_year = season_data.groupby('season_year').count().compute()
        complete_years = months_per_year.where(
            months_per_year == n_months_expected, drop=True
        ).season_year

        season_data = season_data.where(
            season_data.season_year.isin(complete_years), drop=True
        )

        season_mean = season_data.groupby('season_year').mean(dim='time')
        season_mean = season_mean.rename({'season_year': 'year'})

        logger.info(f"    Years: {season_mean.year.values[0]} to {season_mean.year.values[-1]}")
        logger.info(f"    N = {len(season_mean.year)} complete seasons")

        return season_mean


    # ====================================================================
    # MAIN PROCESSING
    # ====================================================================

    log_processing_step(logger, "FIGURE 6: Surface Energy Budget and CAO Index")

    # Load data
    log_data_loading(logger, 'ERA5', str(DATA_DIR / DATA_PATTERN))
    ds_era = load_era5_data()

    # Calculate surface energy budget
    log_processing_step(logger, "Calculating surface energy budget")
    seb_total = calculate_energy_budget(ds_era)

    # Calculate CAO index
    log_processing_step(logger, "Calculating CAO index")
    cao_index = calculate_cao_index(ds_era, PRESSURE_LEVEL)

    # Seasonal aggregation
    log_processing_step(logger, "Aggregating to seasonal means")

    seb_early = calculate_seasonal_means(seb_total, EARLY_WINTER_MONTHS, 'early')
    seb_late = calculate_seasonal_means(seb_total, LATE_WINTER_MONTHS, 'late')

    cao_early = calculate_seasonal_means(cao_index, EARLY_WINTER_MONTHS, 'early')
    cao_late = calculate_seasonal_means(cao_index, LATE_WINTER_MONTHS, 'late')

    # ====================================================================
    # HELPER: STATS BOX LINES
    # ====================================================================

    def sig_str(mk_result):
        if mk_result is None:
            return ''
        p = mk_result['p_value']
        return '**' if p < 0.01 else '*' if p < 0.05 else ''

    def seb_stats_line(season_label, yr_start, yr_end, stats):
        """Format one stats line for SEB: W m⁻² decade⁻¹."""
        if stats is None:
            return None
        slope_per_decade = stats['slope'] * 10
        sig = sig_str(stats['mk_result'])
        return f"{season_label} {yr_start}\u2013{yr_end}: {slope_per_decade:+.2f} W m\u207b\u00b2 decade\u207b\u00b9{sig}"
    
    def cao_stats_line(season_label, yr_start, yr_end, stats):
        """Format one stats line for CAO: K decade⁻¹."""
        if stats is None:
            return None
        slope_per_decade = stats['slope'] * 10
        sig = sig_str(stats['mk_result'])
        return f"{season_label} {yr_start}\u2013{yr_end}: {slope_per_decade:+.3f} K decade\u207b\u00b9{sig}"

    # ====================================================================
    # CREATE FIGURE
    # ====================================================================

    log_processing_step(logger, "Creating 2-panel figure")

    fig, axes = plt.subplots(2, 1, figsize=FIGURE_SIZE_TIMESERIES, sharex=True)

    # ====================================================================
    # PANEL A: SURFACE ENERGY BUDGET
    # ====================================================================

    ax = axes[0]
    logger.info("  Plotting panel (a): Surface energy budget...")

    years_early = seb_early.year.values
    values_early = seb_early.values

    years_late = seb_late.year.values
    values_late = seb_late.values

    ax.plot(years_early, values_early, color=COLOR_EARLY, linewidth=1.5,
           label='Early winter (ONDJ)', alpha=0.8, zorder=3)

    ax.plot(years_late, values_late, color=COLOR_LATE, linewidth=1.5,
           label='Late winter (FMA)', alpha=0.8, zorder=3)

    # Calculate trends
    # TODO: fix mask_early_p1 / mask_late_p1 to use < SPLIT_YEAR (not <=) to avoid
    # double-counting 2015 into both periods. This will alter trend values.
    # Leave display labels as 1979-2015 until mask is corrected and rerun.
    mask_early_p1 = (years_early >= PERIOD1_YEARS[0]) & (years_early <= SPLIT_YEAR)
    stats_early_p1 = get_trend_stats(years_early[mask_early_p1], values_early[mask_early_p1])

    mask_early_p2 = (years_early >= SPLIT_YEAR)
    stats_early_p2 = get_trend_stats(years_early[mask_early_p2], values_early[mask_early_p2])

    mask_late_p1 = (years_late >= PERIOD1_YEARS[0]) & (years_late <= SPLIT_YEAR)
    stats_late_p1 = get_trend_stats(years_late[mask_late_p1], values_late[mask_late_p1])

    mask_late_p2 = (years_late >= SPLIT_YEAR)
    stats_late_p2 = get_trend_stats(years_late[mask_late_p2], values_late[mask_late_p2])

    # Plot trends and CIs
    def plot_trends_ci_on(ax, stats_p1, stats_p2, color):
        if stats_p1:
            ax.plot(stats_p1['years'], stats_p1['trend_line'],
                   '--', color=color, linewidth=2.5, alpha=0.7, zorder=2)
            if stats_p1['bootstrap_ci']:
                ax.fill_between(stats_p1['years'],
                               stats_p1['bootstrap_ci']['ci_lower_band'],
                               stats_p1['bootstrap_ci']['ci_upper_band'],
                               color=color, alpha=0.2, zorder=1)

        if stats_p2:
            ax.plot(stats_p2['years'], stats_p2['trend_line'],
                   '--', color=color, linewidth=2.5, alpha=0.7, zorder=2)
            if stats_p2['bootstrap_ci']:
                ax.fill_between(stats_p2['years'],
                               stats_p2['bootstrap_ci']['ci_lower_band'],
                               stats_p2['bootstrap_ci']['ci_upper_band'],
                               color=color, alpha=0.2, zorder=1)

    plot_trends_ci_on(ax, stats_early_p1, stats_early_p2, COLOR_EARLY)
    plot_trends_ci_on(ax, stats_late_p1, stats_late_p2, COLOR_LATE)

    ax.axvline(x=SPLIT_YEAR, color='gray', linestyle=':', linewidth=1.5, alpha=0.6, zorder=1)

    # Statistics box — per-decade rates, scientific notation units
    if SHOW_STATS:
        stats_lines = []
        l = seb_stats_line('Early', PERIOD1_YEARS[0], SPLIT_YEAR, stats_early_p1)
        if l: stats_lines.append(l)
        l = seb_stats_line('Early', SPLIT_YEAR, PERIOD2_YEARS[1], stats_early_p2)
        if l: stats_lines.append(l)
        l = seb_stats_line('Late ', PERIOD1_YEARS[0], SPLIT_YEAR, stats_late_p1)
        if l: stats_lines.append(l)
        l = seb_stats_line('Late ', SPLIT_YEAR, PERIOD2_YEARS[1], stats_late_p2)
        if l: stats_lines.append(l)

        ax.text(0.03, 0.97, '\n'.join(stats_lines), transform=ax.transAxes,
               fontsize=9, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9),
               family='monospace')

    # ax.axhline(y=0, color='k', linestyle=':', linewidth=1, alpha=0.5)
    ax.set_ylabel('Surface Energy Flux (W m$^{-2}$)', fontsize=11)
    ax.grid(True, alpha=0.3, linewidth=0.5)
    ax.legend(fontsize=10, loc='lower right', framealpha=0.9)
    ax.set_title('(a) Total Surface Energy Budget', fontsize=11, loc='left', fontweight='normal')

    # ====================================================================
    # PANEL B: CAO INDEX
    # ====================================================================

    ax = axes[1]
    logger.info("  Plotting panel (b): CAO index...")

    years_early = cao_early.year.values
    values_early = cao_early.values

    years_late = cao_late.year.values
    values_late = cao_late.values

    ax.plot(years_early, values_early, color=COLOR_EARLY, linewidth=1.5,
           label='Early winter (ONDJ)', alpha=0.8, zorder=3)

    ax.plot(years_late, values_late, color=COLOR_LATE, linewidth=1.5,
           label='Late winter (FMA)', alpha=0.8, zorder=3)

    # Calculate trends
    # TODO: fix mask_early_p1 / mask_late_p1 to use < SPLIT_YEAR (not <=) to avoid
    # double-counting 2015 into both periods. This will alter trend values.
    # Leave display labels as 1979-2015 until mask is corrected and rerun.
    mask_early_p1 = (years_early >= PERIOD1_YEARS[0]) & (years_early <= SPLIT_YEAR)
    stats_early_p1 = get_trend_stats(years_early[mask_early_p1], values_early[mask_early_p1])

    mask_early_p2 = (years_early >= SPLIT_YEAR)
    stats_early_p2 = get_trend_stats(years_early[mask_early_p2], values_early[mask_early_p2])

    mask_late_p1 = (years_late >= PERIOD1_YEARS[0]) & (years_late <= SPLIT_YEAR)
    stats_late_p1 = get_trend_stats(years_late[mask_late_p1], values_late[mask_late_p1])

    mask_late_p2 = (years_late >= SPLIT_YEAR)
    stats_late_p2 = get_trend_stats(years_late[mask_late_p2], values_late[mask_late_p2])

    plot_trends_ci_on(ax, stats_early_p1, stats_early_p2, COLOR_EARLY)
    plot_trends_ci_on(ax, stats_late_p1, stats_late_p2, COLOR_LATE)

    ax.axvline(x=SPLIT_YEAR, color='gray', linestyle=':', linewidth=1.5, alpha=0.6, zorder=1)

    # Statistics box — per-decade rates, scientific notation units
    if SHOW_STATS:
        stats_lines = []
        l = cao_stats_line('Early', PERIOD1_YEARS[0], SPLIT_YEAR, stats_early_p1)
        if l: stats_lines.append(l)
        l = cao_stats_line('Early', SPLIT_YEAR, PERIOD2_YEARS[1], stats_early_p2)
        if l: stats_lines.append(l)
        l = cao_stats_line('Late ', PERIOD1_YEARS[0], SPLIT_YEAR, stats_late_p1)
        if l: stats_lines.append(l)
        l = cao_stats_line('Late ', SPLIT_YEAR, PERIOD2_YEARS[1], stats_late_p2)
        if l: stats_lines.append(l)

        ax.text(0.03, 0.97, '\n'.join(stats_lines), transform=ax.transAxes,
               fontsize=9, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9),
               family='monospace')

    ax.axhline(y=0, color='k', linestyle=':', linewidth=1, alpha=0.5)
    ax.set_ylabel('CAO Index (K)', fontsize=11)
    ax.set_xlabel('Year', fontsize=11)
    ax.grid(True, alpha=0.3, linewidth=0.5)
    ax.legend(fontsize=10, loc='lower right', framealpha=0.9)
    ax.set_title(f'(b) Cold Air Outbreak Index ({PRESSURE_LEVEL} hPa)',
                fontsize=11, loc='left', fontweight='normal')

    # ====================================================================
    # X-AXIS: ticks every 5 years, limits from data
    # ====================================================================
    all_years = np.concatenate([seb_early.year.values, seb_late.year.values,
                                 cao_early.year.values, cao_late.year.values])
    x_min = int(all_years.min()) - 1
    x_max = int(all_years.max()) + 1
    ax.set_xlim(x_min, x_max)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
    # ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

    plt.tight_layout()

    # ====================================================================
    # SAVE OUTPUTS
    # ====================================================================

    log_processing_step(logger, "Saving figure and data")

    filename = 'figure_06_CAOidx_SEB.png'
    fig.savefig(OUTPUT_DIR / filename, dpi=DPI, bbox_inches='tight')
    logger.info(f"  Saved: {filename}")
    log_output_file(logger, 'figure', OUTPUT_DIR / filename)

    plt.close(fig)

    # Save data
    if REPROCESS and SAVE_DATA:
        logger.info("Saving data to CSV...")

        pd.DataFrame({'year': seb_early.year.values, 'seb_total': seb_early.values}).to_csv(
            PROCESSED_DIR / 'surface_energy_budget_early.csv', index=False)
        pd.DataFrame({'year': seb_late.year.values, 'seb_total': seb_late.values}).to_csv(
            PROCESSED_DIR / 'surface_energy_budget_late.csv', index=False)
        pd.DataFrame({'year': cao_early.year.values, 'cao_index': cao_early.values}).to_csv(
            PROCESSED_DIR / 'cao_index_early.csv', index=False)
        pd.DataFrame({'year': cao_late.year.values, 'cao_index': cao_late.values}).to_csv(
            PROCESSED_DIR / 'cao_index_late.csv', index=False)

        for f in ['surface_energy_budget_early.csv', 'surface_energy_budget_late.csv',
                  'cao_index_early.csv', 'cao_index_late.csv']:
            log_output_file(logger, 'processed_data', PROCESSED_DIR / f)

    # ====================================================================
    # GENERATE METHODS DOCUMENTATION
    # ====================================================================

    methods_file = OUTPUT_METHODS_DIR / 'figure_06_methods.md'
    with open(methods_file, 'w') as f:
        f.write(f"""# Figure 6: Methodology

## Surface Energy Budget and Cold Air Outbreak Index

### Data Sources

**ERA5 Reanalysis** (Hersbach et al., 2020)
- **Temporal Coverage**: 1979-2025
- **Spatial Resolution**: 0.25° × 0.25°
- **Temporal Resolution**: Monthly means
- **Region**: Greenland Sea

### Panel A: Surface Energy Budget

#### Budget Equation

The total surface energy flux is calculated as:

F₀ = F_SW,net + F_LW,net + F_SH + F_LH

**Sign convention**: Positive = downward flux (energy into ocean surface)

#### Unit Conversion

ERA5 provides daily accumulated fluxes (J m⁻²) averaged to monthly means.
Conversion to W m⁻²: F = [F_accumulated × days_per_month] / seconds_per_month

#### Spatial Masking

Three masks applied: Greenland Sea polygon, ocean only (lsm = 0),
sea ice concentration < {SIC_THRESHOLD*100:.0f}%.

### Panel B: Cold Air Outbreak (CAO) Index

CAO Index = θ_skin − θ_atmosphere(p)

Default pressure level: {PRESSURE_LEVEL} hPa. Positive values indicate cold air
over warm ocean, driving enhanced turbulent heat fluxes.

### Seasonal Definitions

- **Early Winter (ONDJ)**: October–January (labelled by year of January)
- **Late Winter (FMA)**: February–April

### Statistical Analysis

Split-trend periods: {PERIOD1_YEARS[0]}–{SPLIT_YEAR} and {SPLIT_YEAR}–{PERIOD2_YEARS[1]}.
Breakpoint pre-specified at {SPLIT_YEAR} on physical grounds.
Trend rates reported per decade. Significance: ** (p<0.01), * (p<0.05).
Bootstrap CI: n={N_BOOTSTRAP} resamples, 95% prediction bands.

## References

Hersbach, H., et al. (2020). The ERA5 global reanalysis. *QJRMS*, 146(730), 1999-2049.
""")

    logger.info(f"Methods documentation saved: {methods_file}")
    log_output_file(logger, 'methods', methods_file)

    log_completion(logger, start_time)

    logger.info("="*70)
    logger.info("FIGURE 6 COMPLETE")
    logger.info("="*70)

except Exception as e:
    log_error(logger, e, context="During Figure 6 generation")
    raise

INFO     | ======================================================================
INFO     | Starting analysis: figure_06
INFO     | Log file: outputs/logs/figure_06_20260622_153042.log
INFO     | Timestamp: 2026-06-22 15:30:42
INFO     | ======================================================================
INFO     | Loading configuration...
INFO     | Split year: 2015
INFO     | SIC threshold: 0.15
INFO     | Pressure level: 850 hPa
INFO     | Functions defined
INFO     | Processing: FIGURE 6: Surface Energy Budget and CAO Index
INFO     | Loading ERA5 data from: ../../era5/era5_*_Arctic.nc
INFO     | Loading preprocessed ERA5 data...
INFO     |   Found 47 files
INFO     |   Time range: 1979-01-01T00:00:00.000000000 to 2025-05-01T00:00:00.000000000
INFO     |   All required variables present!
INFO     | Processing: Calculating surface energy budget
INFO     | Calculating surface energy budget...
INFO     |   Converting accumulated fluxes to rates...
INFO     |   Mean total flux: -90